# PCL Classification — BestModel

**Task:** SemEval-2022 Task 4, Subtask 1 — Binary PCL classification  
**Baseline:** RoBERTa-base, 1 epoch, thresh=0.5 → dev F1 = 0.48

## Approach (stacked improvements)
1. **Stronger backbone** — `roberta-large` instead of `roberta-base`
2. **WeightedRandomSampler** — oversample the 9.5% minority PCL class (PALI-NLP #1)
3. **EDA augmentation** — synonym replacement on PCL-positive examples only (CS/NLP SemEval)
4. **FGM adversarial training** — perturb word embeddings for robustness (GUTS SemEval)
5. **Threshold sweep** — optimise decision threshold on dev F1 instead of using 0.5 (UMass)
6. **3-seed ensemble** — average softmax probabilities across seeds (UTSA SemEval)

**Outputs:** `dev.txt` (2094 lines) and `test.txt` (3832 lines) — one 0/1 prediction per line.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers[torch]', 'scikit-learn', 'nltk',
                'pandas', 'numpy', 'matplotlib', 'seaborn'], check=True)

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
print('Installs complete.')

In [ ]:
# ── Cell 2: Clone data repos (idempotent) ─────────────────────────────────────
import os, subprocess

# NLPLabs-2024 contains the training TSV and category files
if not os.path.exists('NLPLabs-2024'):
    subprocess.run(['git', 'clone', 'https://github.com/CRLala/NLPLabs-2024.git'], check=True)
    print('Cloned NLPLabs-2024')
else:
    print('NLPLabs-2024 already present')

# dontpatronizeme contains the helper loader, evaluation script, practice splits & test set
if not os.path.exists('dontpatronizeme'):
    subprocess.run(['git', 'clone', 'https://github.com/Perez-AlmendrosC/dontpatronizeme.git'], check=True)
    print('Cloned dontpatronizeme')
else:
    print('dontpatronizeme already present')

# Verify key files exist
TRAIN_TSV   = 'NLPLabs-2024/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv'
TEST_TSV    = 'dontpatronizeme/semeval-2022/TEST/task4_test.tsv'
TRAIN_IDS   = 'dontpatronizeme/semeval-2022/practice splits/train_semeval_parids-labels.csv'
DEV_IDS     = 'dontpatronizeme/semeval-2022/practice splits/dev_semeval_parids-labels.csv'
HELPER_DIR  = 'dontpatronizeme/semeval-2022'

for f in [TRAIN_TSV, TEST_TSV, TRAIN_IDS, DEV_IDS]:
    assert os.path.exists(f), f'Missing: {f}'
    print(f'  OK: {f}')

In [ ]:
# ── Cell 3: CONFIG — edit these before running ────────────────────────────────
MODEL_NAME     = 'roberta-large'   # swap to 'microsoft/deberta-v3-large' for higher ceiling
N_EPOCHS       = 4                 # early stopping will cut short if dev F1 plateaus
N_SEEDS        = 3                 # ensemble size
BATCH_SIZE     = 16                # reduce to 8 if GPU OOM
GRAD_ACCUM     = 2                 # effective batch = BATCH_SIZE * GRAD_ACCUM
MAX_LEN        = 128               # 95th pct of word counts is 102; 128 tokens is safe
LR             = 2e-5
WARMUP_RATIO   = 0.1
AUG_COPIES     = 1                 # augmented copies per PCL positive example
FGM_EPSILON    = 0.5               # FGM perturbation magnitude
FGM_EMB_NAME   = 'word_embeddings' # layer to perturb (roberta uses 'word_embeddings')
CHECKPOINT_DIR = 'checkpoints'
PATIENCE       = 2                 # stop if no improvement for PATIENCE epochs

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Config: model={MODEL_NAME}, epochs={N_EPOCHS}, seeds={N_SEEDS}, bs={BATCH_SIZE}')

In [ ]:
# ── Cell 4: Imports ───────────────────────────────────────────────────────────
import sys, random, warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

sys.path.insert(0, HELPER_DIR)
from dont_patronize_me import DontPatronizeMe

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 5: Data Loading ──────────────────────────────────────────────────────
TRAIN_PATH = 'NLPLabs-2024/Dont_Patronize_Me_Trainingset'
dpm = DontPatronizeMe(TRAIN_PATH, TEST_TSV)
dpm.load_task1()
full_df = dpm.train_task1_df.copy()   # all 10469 rows: par_id, text, label, orig_label

# Load official train / dev split par_ids
trids = pd.read_csv(TRAIN_IDS)
teids = pd.read_csv(DEV_IDS)
trids.par_id = trids.par_id.astype(str)
teids.par_id = teids.par_id.astype(str)
full_df.par_id = full_df.par_id.astype(str)

# Merge to get text + label for each split
id2row = full_df.set_index('par_id')
train_df = id2row.loc[trids.par_id].reset_index()[['par_id','text','label','keyword']].copy()
dev_df   = id2row.loc[teids.par_id].reset_index()[['par_id','text','label','keyword']].copy()

# Load test set (no labels)
dpm.load_test()
test_df = dpm.test_set_df.copy()
# The load_test() reads all lines including header as first row; filter it out
test_df = test_df[test_df.par_id != 'par_id'].reset_index(drop=True)

print(f'Train: {len(train_df)} rows  |  PCL: {train_df.label.sum()} ({train_df.label.mean()*100:.1f}%)')
print(f'Dev:   {len(dev_df)} rows  |  PCL: {dev_df.label.sum()} ({dev_df.label.mean()*100:.1f}%)')
print(f'Test:  {len(test_df)} rows (unlabelled)')

In [ ]:
# ── Cell 6: EDA Augmentation (synonym replacement, PCL positives only) ────────
import re
from nltk.corpus import wordnet

def get_synonyms(word):
    """Return a set of synonyms from WordNet, excluding the word itself."""
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace('_', ' ')
            if s.lower() != word.lower():
                syns.add(s)
    return syns

def synonym_replace(text, n_replacements=1, seed=None):
    """Replace n random content words with a WordNet synonym."""
    rng = random.Random(seed)
    words = text.split()
    eligible = [(i, w) for i, w in enumerate(words)
                if w.isalpha() and len(w) > 3 and get_synonyms(w.lower())]
    rng.shuffle(eligible)
    replacements = 0
    for idx, word in eligible:
        syns = list(get_synonyms(word.lower()))
        if syns:
            words[idx] = rng.choice(syns)
            replacements += 1
            if replacements >= n_replacements:
                break
    return ' '.join(words)

print('Augmenting PCL-positive training examples...')
pcl_pos = train_df[train_df.label == 1].copy()
aug_rows = []
for _ in range(AUG_COPIES):
    for i, row in pcl_pos.iterrows():
        aug_text = synonym_replace(row['text'], n_replacements=2, seed=i)
        aug_rows.append({'par_id': row['par_id'] + '_aug',
                         'text': aug_text, 'label': 1, 'keyword': row['keyword']})

aug_df = pd.DataFrame(aug_rows)
train_aug_df = pd.concat([train_df, aug_df], ignore_index=True)
print(f'Train after augmentation: {len(train_aug_df)} rows')
print(f'  PCL: {train_aug_df.label.sum()} ({train_aug_df.label.mean()*100:.1f}%)')

In [ ]:
# ── Cell 7: Dataset class + WeightedRandomSampler ─────────────────────────────
class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts  = list(texts)
        self.labels = list(labels) if labels is not None else None
        self.tok    = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def make_weighted_sampler(labels):
    """WeightedRandomSampler: upsamples minority class so each class
    appears roughly equally often in each batch."""
    counts = Counter(labels)
    weights = [1.0 / counts[l] for l in labels]
    return WeightedRandomSampler(
        weights=weights, num_samples=len(weights), replacement=True
    )

print('Dataset and sampler utilities defined.')

In [ ]:
# ── Cell 8: FGM Adversarial Training ─────────────────────────────────────────
class FGM:
    """
    Fast Gradient Method (Miyato et al. 2017) adversarial training.
    Perturbs word embeddings in the direction of the gradient to make
    the model more robust. Used inside the training loop after the
    normal backward pass.

    Reference: GUTS (SemEval-2022 Task 4) reported a clear F1 bump
    from adding FGM to their RoBERTa fine-tuning pipeline.
    """
    def __init__(self, model):
        self.model  = model
        self.backup = {}

    def attack(self, epsilon=1.0, emb_name='word_embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self, emb_name='word_embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                assert name in self.backup
                param.data = self.backup[name]
        self.backup = {}

print('FGM class defined.')

In [ ]:
# ── Cell 9: Training and Evaluation Functions ─────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one_epoch(model, loader, optimizer, scheduler, device, fgm, grad_accum=1):
    """Train one epoch with FGM adversarial training and gradient accumulation.
    Returns mean training loss.
    """
    model.train()
    total_loss, n_batches = 0.0, 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        # ── Normal forward + backward ──
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / grad_accum
        loss.backward()

        # ── FGM adversarial forward + backward ──
        fgm.attack(epsilon=FGM_EPSILON, emb_name=FGM_EMB_NAME)
        adv_outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        adv_loss = adv_outputs.loss / grad_accum
        adv_loss.backward()
        fgm.restore(emb_name=FGM_EMB_NAME)

        total_loss += outputs.loss.item()
        n_batches  += 1

        # ── Gradient accumulation step ──
        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader, device):
    """Run inference. Returns (softmax_probs [N,2], labels [N]) numpy arrays.
    If loader has no labels, the labels array will be None.
    """
    model.eval()
    all_probs, all_labels = [], []
    has_labels = 'labels' in next(iter(loader))

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs  = torch.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())
        if has_labels:
            all_labels.append(batch['labels'].numpy())

    probs_arr  = np.vstack(all_probs)                                   # shape (N, 2)
    labels_arr = np.concatenate(all_labels) if all_labels else None     # shape (N,)
    return probs_arr, labels_arr


print('Training and evaluation functions defined.')

In [ ]:
# ── Cell 10: Threshold Sweep ──────────────────────────────────────────────────
def find_best_threshold(probs_pos, labels, lo=0.05, hi=0.95, step=0.01):
    """
    Sweep decision threshold from lo to hi and return the one maximising
    F1 on the positive class (PCL=1).

    Args:
        probs_pos: 1-D array of P(PCL=1) for each example
        labels:    true binary labels (0/1)
    Returns:
        (best_threshold, best_f1)
    """
    best_f1, best_thresh = 0.0, 0.5
    thresholds = np.arange(lo, hi + step, step)
    for thresh in thresholds:
        preds = (probs_pos >= thresh).astype(int)
        f1 = f1_score(labels, preds, pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    return float(best_thresh), float(best_f1)

print('Threshold sweep function defined.')

In [ ]:
# ── Cell 11: Main Training Loop ───────────────────────────────────────────────
# Trains N_SEEDS models. After every epoch: saves checkpoint + prints dev F1.
# Best epoch per seed is saved separately. Collects probs for ensemble.

all_dev_probs  = []   # shape: (N_SEEDS, n_dev)
all_test_probs = []   # shape: (N_SEEDS, n_test)
dev_labels_ref = None # will be set on first seed

for seed in range(N_SEEDS):
    print(f'\n{'='*60}')
    print(f'SEED {seed}  |  Model: {MODEL_NAME}')
    print(f'{'='*60}')

    set_seed(seed)

    # ── Tokenizer + DataLoaders ──
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_dataset = PCLDataset(train_aug_df.text.tolist(),
                               train_aug_df.label.tolist(),
                               tokenizer, MAX_LEN)
    dev_dataset   = PCLDataset(dev_df.text.tolist(),
                               dev_df.label.tolist(),
                               tokenizer, MAX_LEN)
    test_dataset  = PCLDataset(test_df.text.tolist(),
                               None,           # no labels for test
                               tokenizer, MAX_LEN)

    sampler = make_weighted_sampler(train_aug_df.label.tolist())
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              sampler=sampler, num_workers=2, pin_memory=True)
    dev_loader   = DataLoader(dev_dataset,  batch_size=BATCH_SIZE * 2,
                              shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2,
                              shuffle=False, num_workers=2, pin_memory=True)

    # ── Model + optimiser + scheduler ──
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=LR, eps=1e-8, weight_decay=0.01)

    n_training_steps = (len(train_loader) // GRAD_ACCUM) * N_EPOCHS
    n_warmup_steps   = int(n_training_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=n_warmup_steps,
        num_training_steps=n_training_steps
    )

    fgm = FGM(model)

    # ── Epoch loop ──
    best_dev_f1  = 0.0
    epochs_no_improve = 0

    for epoch in range(N_EPOCHS):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scheduler,
            device, fgm, grad_accum=GRAD_ACCUM
        )

        dev_probs, dev_labels = evaluate(model, dev_loader, device)
        thresh, dev_f1 = find_best_threshold(dev_probs[:, 1], dev_labels)

        print(f'  Epoch {epoch+1}/{N_EPOCHS} | '
              f'train_loss={train_loss:.4f} | '
              f'dev_F1={dev_f1:.4f} @ thresh={thresh:.2f}')

        # ── Save checkpoint every epoch ──
        ckpt_path = f'{CHECKPOINT_DIR}/seed{seed}_epoch{epoch+1}'
        model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        print(f'    Checkpoint saved → {ckpt_path}')

        # ── Track best ──
        if dev_f1 > best_dev_f1:
            best_dev_f1 = dev_f1
            best_path   = f'{CHECKPOINT_DIR}/seed{seed}_best'
            model.save_pretrained(best_path)
            tokenizer.save_pretrained(best_path)
            print(f'    ★ New best dev F1={dev_f1:.4f} — saved → {best_path}')
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f'    Early stopping after {epoch+1} epochs (patience={PATIENCE})')
                break

    # ── Collect probs from best checkpoint for ensemble ──
    print(f'  Loading best checkpoint for seed {seed}...')
    best_model = AutoModelForSequenceClassification.from_pretrained(best_path).to(device)
    dev_probs_best, dev_labels_best   = evaluate(best_model, dev_loader, device)
    test_probs_best, _                = evaluate(best_model, test_loader, device)
    all_dev_probs.append(dev_probs_best[:, 1])
    all_test_probs.append(test_probs_best[:, 1])
    if dev_labels_ref is None:
        dev_labels_ref = dev_labels_best
    del best_model
    torch.cuda.empty_cache()
    print(f'  Seed {seed} best dev F1 = {best_dev_f1:.4f}')

print('\nAll seeds trained.')

In [ ]:
# ── Cell 12: Ensemble + Final Threshold Sweep ─────────────────────────────────
ensemble_dev_probs  = np.mean(all_dev_probs,  axis=0)   # shape: (n_dev,)
ensemble_test_probs = np.mean(all_test_probs, axis=0)   # shape: (n_test,)

# Per-seed F1 for reference
print('Per-seed dev F1 (best checkpoint, optimal threshold):')
for i, dp in enumerate(all_dev_probs):
    t, f = find_best_threshold(dp, dev_labels_ref)
    print(f'  Seed {i}: F1={f:.4f} @ thresh={t:.2f}')

# Ensemble F1
best_thresh, best_f1 = find_best_threshold(ensemble_dev_probs, dev_labels_ref)
print(f'\nEnsemble dev F1 = {best_f1:.4f} @ threshold = {best_thresh:.2f}')
print(f'Baseline dev F1 = 0.48  →  delta = {best_f1 - 0.48:+.4f}')

dev_preds  = (ensemble_dev_probs  >= best_thresh).astype(int)
test_preds = (ensemble_test_probs >= best_thresh).astype(int)

print(f'\nDev  predictions: {Counter(dev_preds)}')
print(f'Test predictions: {Counter(test_preds)}')

In [ ]:
# ── Cell 13: Save dev.txt + test.txt ─────────────────────────────────────────
with open('dev.txt', 'w') as f:
    for p in dev_preds:
        f.write(str(int(p)) + '\n')

with open('test.txt', 'w') as f:
    for p in test_preds:
        f.write(str(int(p)) + '\n')

# Verify line counts
dev_lines  = sum(1 for _ in open('dev.txt'))
test_lines = sum(1 for _ in open('test.txt'))
print(f'dev.txt  : {dev_lines:5d} lines  (expected 2094)')
print(f'test.txt : {test_lines:5d} lines  (expected 3832)')
assert dev_lines  == len(dev_df),  f'dev.txt line count mismatch: {dev_lines} vs {len(dev_df)}'
print('\nOutput files saved successfully.')

---
## Local Evaluation (Exercise 5.2)
Classification report, confusion matrix, precision-recall curve, and error analysis.

In [ ]:
# ── Cell 14: Classification Report + Confusion Matrix ────────────────────────
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score
)

print('=== Classification Report (dev set, ensemble) ===')
print(classification_report(dev_labels_ref, dev_preds,
                            target_names=['No PCL (0)', 'PCL (1)'], digits=4))

# Confusion matrix
cm = confusion_matrix(dev_labels_ref, dev_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: No PCL', 'Pred: PCL'],
            yticklabels=['True: No PCL', 'True: PCL'], ax=ax)
ax.set_title(f'Confusion Matrix (dev set)\nEnsemble F1={best_f1:.4f} @ thresh={best_thresh:.2f}',
             fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved confusion_matrix.png')

In [ ]:
# ── Cell 15: Precision-Recall Curve ──────────────────────────────────────────
precision, recall, pr_thresholds = precision_recall_curve(dev_labels_ref, ensemble_dev_probs)
ap = average_precision_score(dev_labels_ref, ensemble_dev_probs)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision, color='#DD8452', linewidth=2, label=f'PR curve (AP={ap:.3f})')
ax.axvline(recall[(np.abs(pr_thresholds - best_thresh)).argmin()],
           color='black', linestyle='--', alpha=0.5, label=f'Chosen threshold={best_thresh:.2f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve (dev set, PCL class)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pr_curve.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved pr_curve.png')

In [ ]:
# ── Cell 16: Error Analysis ───────────────────────────────────────────────────
dev_df_eval = dev_df.reset_index(drop=True).copy()
dev_df_eval['prob_pcl']   = ensemble_dev_probs
dev_df_eval['pred']       = dev_preds
dev_df_eval['true']       = dev_labels_ref
dev_df_eval['correct']    = dev_df_eval['pred'] == dev_df_eval['true']

# False Positives: predicted PCL but actually No-PCL (high confidence wrong)
fp = dev_df_eval[(dev_df_eval.pred==1) & (dev_df_eval.true==0)].sort_values('prob_pcl', ascending=False)
# False Negatives: predicted No-PCL but actually PCL
fn = dev_df_eval[(dev_df_eval.pred==0) & (dev_df_eval.true==1)].sort_values('prob_pcl', ascending=True)

print(f'False Positives (predicted PCL, actually No-PCL): {len(fp)}')
print(f'False Negatives (predicted No-PCL, actually PCL): {len(fn)}')
print()

print('=== Top 5 False Positives (most confident wrong PCL predictions) ===')
for _, row in fp.head(5).iterrows():
    print(f'  P(PCL)={row.prob_pcl:.3f} | keyword={row.keyword}')
    print(f'  Text: {row.text[:200]}...')
    print()

print('=== Top 5 False Negatives (missed PCL — lowest confidence) ===')
for _, row in fn.head(5).iterrows():
    print(f'  P(PCL)={row.prob_pcl:.3f} | keyword={row.keyword}')
    print(f'  Text: {row.text[:200]}...')
    print()

# Error rate by keyword
print('=== Error Rate by Keyword (dev set) ===')
err_by_kw = dev_df_eval.groupby('keyword').apply(
    lambda g: pd.Series({
        'total': len(g),
        'FP': ((g.pred==1) & (g.true==0)).sum(),
        'FN': ((g.pred==0) & (g.true==1)).sum(),
        'PCL_F1': f1_score(g.true, g.pred, pos_label=1, zero_division=0)
    })
).sort_values('PCL_F1')
print(err_by_kw.to_string())

In [ ]:
# ── Cell 17: Summary Table ────────────────────────────────────────────────────
print('=== Final Results Summary ===')
print(f'Model:              {MODEL_NAME}')
print(f'Epochs trained:     up to {N_EPOCHS} (early stopping patience={PATIENCE})')
print(f'Seeds in ensemble:  {N_SEEDS}')
print(f'Augmentation:       {AUG_COPIES}x synonym replacement on PCL positives')
print(f'FGM epsilon:        {FGM_EPSILON}')
print(f'Decision threshold: {best_thresh:.2f} (swept on dev)')
print()
print(f'Baseline dev F1:    0.48')
print(f'Our dev F1:         {best_f1:.4f}  (delta = {best_f1-0.48:+.4f})')
print()
print(f'dev.txt lines:      {dev_lines}')
print(f'test.txt lines:     {test_lines}')